# EDA — WDR91 DEL Screen Data

DREAM Target 2035 Challenge · Step 1: Retrospective Validation

**Dataset:** `data/raw/WDR91.parquet` — 375,595 DEL compounds screened against WDR91

**Goal:** Understand the DEL training data before building hit-retrieval models.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

DATA_PATH = '../data/raw/WDR91.parquet'

## 1. Load Scalar Columns

Fingerprint columns are stored as sparse int arrays and are heavy (~869MB total). We load scalar columns first.

In [ ]:
SCALAR_COLS = [
    'COMPOUND_ID', 'LIBRARY_ID', 'BB1_ID', 'BB2_ID', 'BB3_ID',
    'TARGET_ID', 'TARGET_VALUE', 'NTC_VALUE', 'ENRICHMENT',
    'LABEL', 'MW', 'ALOGP'
]

table = pq.read_table(DATA_PATH, columns=SCALAR_COLS)
df = table.to_pandas()

# Derived columns
df['lib_prefix'] = df['LIBRARY_ID'].str[:3]
df['log1p_target'] = np.log1p(df['TARGET_VALUE'])

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2. Label Distribution

In [ ]:
label_counts = df['LABEL'].value_counts()
n_total = len(df)
n_hits = (df['LABEL'] == 1).sum()
n_nonhits = (df['LABEL'] == 0).sum()

print(f'Total compounds  : {n_total:>10,}')
print(f'Hits   (LABEL=1) : {n_hits:>10,}  ({100*n_hits/n_total:.2f}%)')
print(f'Non-hits (LABEL=0): {n_nonhits:>10,}  ({100*n_nonhits/n_total:.2f}%)')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Non-hits\n(LABEL=0)', 'Hits\n(LABEL=1)'],
               [n_nonhits, n_hits],
               color=['steelblue', 'coral'], edgecolor='white')
for bar, val in zip(bars, [n_nonhits, n_hits]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'{val:,}', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Count')
ax.set_title('Label Distribution (DEL Screen)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(FIGURES / 'label_distribution.png', dpi=150)
plt.show()

## 3. TARGET_VALUE Distribution

`TARGET_VALUE` is the raw DEL count per compound. Non-hits have exactly 0; hits range 6–696.

In [ ]:
hits = df[df['LABEL'] == 1]
nonhits = df[df['LABEL'] == 0]

print('TARGET_VALUE by LABEL:')
print(df.groupby('LABEL')['TARGET_VALUE'].describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: full distribution (log scale)
axes[0].hist(hits['TARGET_VALUE'], bins=80, color='coral', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('TARGET_VALUE (hit compounds only)')
axes[0].set_ylabel('Count')
axes[0].set_title('Hit TARGET_VALUE Distribution')
axes[0].set_yscale('log')

# Right: log1p transformed
axes[1].hist(hits['log1p_target'], bins=60, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('log1p(TARGET_VALUE)')
axes[1].set_ylabel('Count')
axes[1].set_title('log1p(TARGET_VALUE) — Hits')

plt.suptitle('DEL Count Distribution for Hits', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'target_value_distribution.png', dpi=150)
plt.show()

print(f'\nHit TARGET_VALUE percentiles:')
print(hits['TARGET_VALUE'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

## 4. DEL Library Breakdown

There are 39 sub-libraries. Which libraries produce the most hits?

In [ ]:
lib_stats = df.groupby('lib_prefix').agg(
    total=('LABEL', 'count'),
    hits=('LABEL', 'sum')
).reset_index()
lib_stats['hit_rate'] = lib_stats['hits'] / lib_stats['total']
lib_stats = lib_stats.sort_values('total', ascending=False)

print(f'Number of DEL sub-libraries: {lib_stats.shape[0]}')
print(lib_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Library size
axes[0].bar(lib_stats['lib_prefix'], lib_stats['total'], color='steelblue', edgecolor='white')
axes[0].set_xlabel('DEL Library')
axes[0].set_ylabel('Compound Count')
axes[0].set_title('Compounds per DEL Library')
axes[0].tick_params(axis='x', rotation=90)

# Hit rate per library
lib_sorted_hr = lib_stats.sort_values('hit_rate', ascending=False)
colors = ['coral' if r > lib_stats['hit_rate'].mean() else 'steelblue' for r in lib_sorted_hr['hit_rate']]
axes[1].bar(lib_sorted_hr['lib_prefix'], lib_sorted_hr['hit_rate'], color=colors, edgecolor='white')
axes[1].axhline(lib_stats['hit_rate'].mean(), color='red', linestyle='--', label=f'Mean ({lib_stats["hit_rate"].mean():.3f})')
axes[1].set_xlabel('DEL Library')
axes[1].set_ylabel('Hit Rate')
axes[1].set_title('Hit Rate per DEL Library')
axes[1].tick_params(axis='x', rotation=90)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'library_breakdown.png', dpi=150)
plt.show()

## 5. Building Block Analysis

DEL compounds are built from 3 building blocks (BB1, BB2, BB3). We look for enriched BBs.

In [ ]:
print(f'Unique BB1: {df["BB1_ID"].nunique():,}')
print(f'Unique BB2: {df["BB2_ID"].nunique():,}')
print(f'Unique BB3: {df["BB3_ID"].nunique():,}')

def top_enriched_bb(df, bb_col, top_n=20):
    stats = df.groupby(bb_col).agg(
        total=('LABEL', 'count'),
        hits=('LABEL', 'sum')
    ).reset_index()
    stats['hit_rate'] = stats['hits'] / stats['total']
    # Only BBs with at least 10 appearances
    stats = stats[stats['total'] >= 10]
    return stats.sort_values('hit_rate', ascending=False).head(top_n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, title in zip(axes, ['BB1_ID', 'BB2_ID', 'BB3_ID'], ['BB1', 'BB2', 'BB3']):
    top = top_enriched_bb(df, col)
    ax.barh(top[col][::-1], top['hit_rate'][::-1], color='coral', edgecolor='white')
    ax.set_xlabel('Hit Rate')
    ax.set_title(f'Top 20 Enriched {title} Building Blocks')
    ax.set_ylabel(col)

plt.tight_layout()
plt.savefig(FIGURES / 'building_block_enrichment.png', dpi=150)
plt.show()

## 6. Physicochemical Properties (Lipinski / Drug-likeness)

In [ ]:
print('MW stats:')
print(df.groupby('LABEL')['MW'].describe())
print('\nALOGP stats:')
print(df.groupby('LABEL')['ALOGP'].describe())

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# MW distributions
for label, color, name in [(0, 'steelblue', 'Non-hits'), (1, 'coral', 'Hits')]:
    subset = df[df['LABEL'] == label]
    axes[0,0].hist(subset['MW'], bins=60, alpha=0.6, color=color, label=name, density=True, edgecolor='white')
axes[0,0].set_xlabel('Molecular Weight (Da)')
axes[0,0].set_ylabel('Density')
axes[0,0].set_title('MW Distribution')
axes[0,0].legend()
axes[0,0].axvline(500, color='black', linestyle='--', alpha=0.5, label='Lipinski MW=500')

# ALogP distributions
for label, color, name in [(0, 'steelblue', 'Non-hits'), (1, 'coral', 'Hits')]:
    subset = df[df['LABEL'] == label]
    axes[0,1].hist(subset['ALOGP'], bins=60, alpha=0.6, color=color, label=name, density=True, edgecolor='white')
axes[0,1].set_xlabel('ALogP')
axes[0,1].set_ylabel('Density')
axes[0,1].set_title('ALogP Distribution')
axes[0,1].legend()
axes[0,1].axvline(5, color='black', linestyle='--', alpha=0.5)

# MW vs ALogP scatter (sample)
sample_hits = hits.sample(min(5000, len(hits)), random_state=42)
sample_nonhits = nonhits.sample(5000, random_state=42)
axes[1,0].scatter(sample_nonhits['MW'], sample_nonhits['ALOGP'], alpha=0.2, s=5, color='steelblue', label='Non-hits')
axes[1,0].scatter(sample_hits['MW'], sample_hits['ALOGP'], alpha=0.4, s=8, color='coral', label='Hits')
axes[1,0].set_xlabel('MW (Da)')
axes[1,0].set_ylabel('ALogP')
axes[1,0].set_title('MW vs ALogP (sampled)')
axes[1,0].legend(markerscale=3)

# MW boxplot
axes[1,1].boxplot([nonhits['MW'], hits['MW']], labels=['Non-hits', 'Hits'],
                  patch_artist=True,
                  boxprops=dict(facecolor='steelblue', alpha=0.6),
                  medianprops=dict(color='red', linewidth=2))
axes[1,1].set_ylabel('MW (Da)')
axes[1,1].set_title('MW Boxplot by Label')

plt.suptitle('Physicochemical Properties: Hits vs Non-hits', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / 'physicochemical_properties.png', dpi=150)
plt.show()

## 7. Fingerprint Availability & Sparse Format Check

In [ ]:
FP_COLS = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']

# Read just one fingerprint for a few rows to understand storage format
pf = pq.ParquetFile(DATA_PATH)
sample_batch = next(pf.iter_batches(batch_size=5, columns=['COMPOUND_ID', 'LABEL', 'ECFP4', 'ECFP6', 'MACCS']))
sample_fp = sample_batch.to_pandas()

print('ECFP4 sample (stored as indices of set bits):')
for i, row in sample_fp.iterrows():
    print(f'  {row["COMPOUND_ID"]} | LABEL={row["LABEL"]} | ECFP4 bits (first 10): {row["ECFP4"][:10]}')

print(f'\nECFP4 bit indices per molecule (nnz): {[len(x) for x in sample_fp["ECFP4"]]}')
print(f'ECFP6 bit indices per molecule (nnz): {[len(x) for x in sample_fp["ECFP6"]]}')
print(f'MACCS  bit indices per molecule (nnz): {[len(x) for x in sample_fp["MACCS"]]}')

print(f'\n=> Fingerprints are stored as sparse lists of ON-bit indices.')
print(f'   Use max index to infer bit vector length.')

# Check max bit index across a batch
big_batch = next(pf.iter_batches(batch_size=10000, columns=['ECFP4', 'ECFP6', 'MACCS']))
big_df = big_batch.to_pandas()
ecfp4_max = max(max(x) if len(x) > 0 else 0 for x in big_df['ECFP4'])
ecfp6_max = max(max(x) if len(x) > 0 else 0 for x in big_df['ECFP6'])
maccs_max  = max(max(x) if len(x) > 0 else 0 for x in big_df['MACCS'])
print(f'\nMax bit index (from 10k sample):')
print(f'  ECFP4: {ecfp4_max}  (standard = 2048 bits)')
print(f'  ECFP6: {ecfp6_max}  (standard = 2048 bits)')
print(f'  MACCS: {maccs_max}  (standard = 167 bits)')

## 8. Missing Values & Data Quality

In [ ]:
print('=== Null counts (scalar columns) ===')
print(df[SCALAR_COLS].isnull().sum().to_string())
print(f'\nSMILES null: {df["COMPOUND_ID"].isnull().sum()} (SMILES column appears null for all — compounds identified by building blocks)')
print('NTC_VALUE: completely null — no-target control not provided in this release')
print('ENRICHMENT: completely null — raw counts only')
print('\n=> No missing values in key columns (LABEL, TARGET_VALUE, MW, ALOGP, BBs).')

## 9. Class Balance — Implication for Modeling

In [ ]:
hit_rate = df['LABEL'].mean()
imbalance_ratio = (1 - hit_rate) / hit_rate

print(f'Overall hit rate   : {hit_rate:.4f} ({100*hit_rate:.2f}%)')
print(f'Imbalance ratio    : {imbalance_ratio:.1f}:1  (non-hits per hit)')
print(f'\nImplication: Use scale_pos_weight={imbalance_ratio:.1f} in XGBoost,')
print(f'             class_weight="balanced" in sklearn models.')
print(f'             Use AUPRC and Enrichment Factor (not accuracy) as metrics.')

# Per-library hit rate spread
lib_hr = df.groupby('lib_prefix')['LABEL'].mean()
print(f'\nHit rate range across libraries: {lib_hr.min():.4f} – {lib_hr.max():.4f}')
print(f'This means library-aware training/splitting is important to avoid leakage.')

## 10. EDA Summary

| Aspect | Finding |
|--------|---------|
| Total compounds | 375,595 |
| Hits (LABEL=1) | 28,778 (7.66%) |
| DEL sub-libraries | 39 |
| Building blocks | BB1: 2,485 · BB2: 2,319 · BB3: 3,562 |
| TARGET_VALUE | 0 for non-hits; 6–696 for hits (median=9) |
| MW range | 253–786 Da (mean ≈ 490) — slightly above Ro5 |
| ALogP range | −3.7 to 9.1 (mean ≈ 3.6) |
| Fingerprints | 9 types pre-computed, stored as sparse bit-index lists |
| Missing data | SMILES/NTC/ENRICHMENT null; all model-relevant columns complete |

### Next Steps
1. Convert sparse fingerprint lists → dense numpy arrays (batch-wise)
2. Train baseline classifiers (XGBoost, RF) on ECFP4/ECFP6
3. Evaluate with Enrichment Factor @ 1%, AUROC, AUPRC
4. Library-aware cross-validation to prevent leakage
5. Apply best model to predict on prospective 4.4M compound library